# Model Building & Evaluation for Recommendation System

## Objective
Build and evaluate recommendation models using collaborative filtering techniques to predict user-product interactions.

## Models to Build
1. **SVD (Singular Value Decomposition)** - For explicit ratings
2. **Implicit ALS** - For implicit feedback (clicks, cart, purchase)

## Libraries Used
- **scikit-surprise**: Collaborative filtering algorithms
- **implicit**: Fast implicit feedback matrix factorization
- **scikit-learn**: Additional preprocessing and evaluation
- **MLflow**: Experiment tracking and model management

## Data Source
- Input: `final_dataset_transformed.csv` (27,000+ records with engineered features)
- Recommendation Matrix: User-Product interactions
- Target: Predict user-product interaction probability

## Model Evaluation
- RMSE: Prediction accuracy
- Precision@K, Recall@K: Ranking quality
- NDCG: Relevance ranking
- Coverage: Catalog coverage

In [ ]:
#importing libraries 
import numpy as np 
import pandas as pd 
import warnings 
warnings.filterwarnings('ignore')

#MLFlow
import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from mlflow.models import infer_signature

# Scikit Surprise (for collaborative filtering)
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise.model_selection import cross_validate, GridSearchCV
from surprise import accuracy

#Implicit (Implicit Feedback)
import implicit
from implicit.als import AlternatingLeastSquares
from implicit.evaluation import ranking_metrics_at_k
from implicit.nearest_neighbours import ItemItemRecommender
from scipy.sparse import csr_matrix

#scikit-learn
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, precision_score, recall_score

#plotting
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

#utilities
import pickle 
import json
from datetime import datetime 
import os

In [ ]:
random_state = 42 
np.random.seed(random_state)

In [ ]:
mlflow.set_experiment("Recommendation Sytem")
print(f"\n ML Flow Experiment: Recommendation System")

## Data Loading and Preparation

### Steps:
1. Load the feature-engineered dataset
2. Create user-product interaction matrix
3. Extract explicit feedback (ratings/scores)
4. Extract implicit feedback (clicks, cart, purchase)
5. Prepare data formats for each model type

### Data Structures:
- **Explicit Ratings**: DataFrame with user_id, product_id, rating
- **Implicit Feedback**: Sparse matrix of user-product interactions
- **Train/Test Split**: 80% train, 20% test

### Handling Interactions:
- **View**: Weight 1.0
- **Cart**: Weight 2.0
- **Purchase**: Weight 3.0 (or explicit rating)
- **Rating**: Direct value

In [ ]:
#loading feature-engineered dataset
df = pd.read_csv(r"e:/dm4ml project/data/processed/final_dataset_transformed.csv")

In [ ]:
df.shape

In [ ]:
print(f"Columns: {df.columns.to_list()[:10]}... (showing first 10)")

In [ ]:
print(f"Data Types: \n {df.dtypes.head(10)}")

In [ ]:
#check for required columns
required_cols = ['user_id', 'product_id', 'interaction_type', 'user_rating', 'timestamp']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols :
    print(f"    Missing Columns: {missing_cols}")
else:
    print(f"    All required Cols available here ")

In [ ]:
print("\n Creating Explicit Ratings Data (for SVD)")

#Extracting explicit rating or derive from interactions
df_explicit = df[['user_id', 'product_id', 'user_rating']].copy()
df_explicit = df_explicit.rename(columns={'user_rating': 'rating'})

#Handling Missing ratings - create from weighted interactions
if df_explicit['rating'].isna().sum() > 0:
    print(f"  {df_explicit['rating'].isna().sum()} missing values")

    #creating weighted ratings from interaction types
    interaction_weights = {
        'view': 1.0,
        'cart': 2.0,
        'purchase': 4.0
    }

    missing_mask = df_explicit['rating']
    for i in df_explicit[missing_mask].index:
        interaction_type = df.loc[i, 'interaction_type']
        df_explicit.loc[i, 'rating'] = interaction_weights.get(interaction_type, 1.0)
        
    print(f"     Filled {df_explicit['rating'].isna().sum()} ratings with weighted interactions")


In [ ]:
df_explicit['rating'] = df_explicit['rating'].clip(1, 5)

#removing duplicates - keeps avg ratings per user-product pair
df_explicit_agg = df_explicit.groupby(['user_id', 'product_id'])['rating'].mean().reset_index()
print(f"Aggregated to {len(df_explicit_agg)} unique user-product ratings pairs")

In [ ]:
df_explicit_agg.describe()

In [ ]:
#creating implicit feedback data (for ALS)

print("\n Creating Implicit Feedback Data (for ALS)")

#creating interaction strength scores
interaction_strength = {
    'view': 1,
    'cart': 3,
    'purchase': 5
}

df_implicit = df[['user_id', 'product_id', 'interaction_type']].copy()
df_implicit['strength'] = df_implicit['interaction_type'].map(interaction_strength)

#aggregating strengths for user-product pairs
df_implicit_agg = df_implicit.groupby(['user_id', 'product_id'])['strength'].sum().reset_index()
df_implicit_agg.rename(columns={'strength': 'confidence'}, inplace=True)

print(f"    Created {len(df_implicit_agg)} implicit feedback records")
print(f"    COnfidence Score Distribution: {df_implicit_agg['confidence'].describe()}")

In [ ]:
print("Creating Sparse interaction matrix")

#ger unique users and product ids
user_ids = sorted(df['user_id'].unique())
product_ids = sorted(df['product_id'].unique())

print(f"Unique Users: {len(user_ids)}")
print(f"Unique Products: {len(product_ids)}")

In [ ]:
#creating mapping dictionaries 
user_id_map = {uid: i for i, uid in enumerate(user_ids)}
product_id_map = {pid: i for i, pid in enumerate(product_ids)}

#creating sparse matrix for ALS
row = df_implicit_agg['user_id'].map(user_id_map).values
col = df_implicit_agg['product_id'].map(product_id_map).values 
data = df_implicit_agg['confidence'].values 

interaction_matrix = csr_matrix((data, (row, col)),
                                    shape=(len(user_ids), len(product_ids)))

print(f" Interaction Matrix Shape: {interaction_matrix.shape}")
print(f" Sparsity: {1 - (interaction_matrix.nnz / (interaction_matrix.shape[0] * interaction_matrix.shape[1]))*100:.2f}%")

In [ ]:
#storing mappings for later use
data_config = {
    'user_ids': user_ids,
    'product_ids': product_ids,
    'user_id_map': user_id_map,
    'product_id_map': product_id_map
}

print("Data preparation complete. Ready for model building and evaluation.")

## SVD Model (Singular Value Decomposition)

### Why SVD?
- Matrix factorization technique for explicit feedback
- Decomposes user-item matrix into latent factors
- Captures hidden patterns in user preferences
- Highly efficient and scalable
- Industry-standard for recommendation systems

### Parameters:
- **n_factors**: Number of latent dimensions (typically 20-100)
- **n_epochs**: Training iterations (typically 20-50)
- **lr_all**: Learning rate for all parameters
- **reg_all**: Regularization factor to prevent overfitting

### Expected Performance:
- RMSE: 0.8-1.2 (lower is better)
- Coverage: High (can recommend most items)
- Training Time: Fast (minutes for large datasets)

In [ ]:
print("Model 1: SVD (Explicit Ratings)")

print("Preparing data for SVD...")

reader = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(df_explicit_agg, reader)

print("Surprise dataset created with SVD")
print(f"Total ratings: {len(surprise_data.raw_ratings)}")

trainset, testset = surprise_train_test_split(surprise_data, test_size=0.2, random_state=random_state) 

print(f" Trainset: {trainset.n_ratings} ratings")
print(f" Testset: {len(testset)} ratings")

In [ ]:
print("Training SVD model...")

with mlflow.start_run(run_name="SVD Model") as run:

    #svd hyperparameters
    svd_params = {
        'n_factors': 50,
        'n_epochs': 30,
        'lr_all': 0.005,
        'reg_all': 0.02,
        'random_state': random_state,
        'verbose': True
    }

    #log parameters to MLFLow
    mlflow.log_params(svd_params)

    #train SVD model
    svd_model = SVD(**svd_params)
    svd_model.fit(trainset)

    print("Evaluating SVD model...")
    print("SVD Model trained")
    print("Model Params logged to MLFlow")

    print(" Evaluating SVD model on testset")

    #predictions on testset
    svd_predictions = svd_model.test(testset)

    #RMSE Calculations
    rmse = accuracy.rmse(svd_predictions, verbose=False)
    mae = accuracy.mae(svd_predictions, verbose=False)

    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")


    #logging metrics to MLFLow
    mlflow.log_metrics({
        'RMSE': rmse,
        'MAE': mae
    })

    print("Calculating Precision@K and Recall@K")

    pred_df = pd.DataFrame([
        {'user': p.uid, 'product':  p.iid, 'estimate': p.est, 'actual': p.r_ui}
        for p in svd_predictions
    ])

    #Capturing Top-K value predictions per user
    K_VALUES = [5, 10, 20]
    precision_at_k = {}
    recall_at_k = {}

    for k in K_VALUES:
        top_k_preds = pred_df.groupby('user').apply(
            lambda x: x.nlargest(k, 'estimate')
        ).reset_index(drop=True)

        # Calculating Precision@K: relavent items / K 
        # (Considering items with actual rating >= 3 as relevant)
        relevant = (top_k_preds['actual'] >= 3).sum()
        precision = relevant / (k * len(pred_df['user'].unique()))

        # Calculationg recall: relevant items at top k / total relavent items
        total_relevant = (pred_df['actual']>=3).sum()
        recall = relevant / total_relevant if total_relevant > 0 else 0

        precision_at_k[f'Precision@{k}'] = precision
        recall_at_k[f'Recall@{k}'] = recall

        print(f"Precision@{k}: {precision:.4f}")
        print(f"Recall@{k}: {recall:.4f}")

    # Sanitize metric names to replace '@' with '_' (MLFlow doesn't allow @ in metric names)
    precision_at_k_sanitized = {k.replace('@', '_'): v for k, v in precision_at_k.items()}
    recall_at_k_sanitized = {k.replace('@', '_'): v for k, v in recall_at_k.items()}
    
    mlflow.log_metrics({**precision_at_k_sanitized, **recall_at_k_sanitized})

    mlflow.sklearn.log_model(svd_model, "svd_model", signature=infer_signature(trainset.build_testset()))
    print("SVD model logged to MLFlow")

print("SVD Model training and evaluation complete.\n")        

## Implicit ALS Model (Alternating Least Squares)

### Why Implicit ALS?
- Designed specifically for implicit feedback
- Handles "non-interaction" as negative feedback
- Efficient even with extreme sparsity
- Industry standard for clickstream data
- Faster training than explicit feedback methods

### Parameters:
- **factors**: Number of latent factors (typically 50-100)
- **iterations**: Training iterations (typically 15-50)
- **regularization**: L2 regularization strength
- **alpha**: Weight of implicit observations
- **calculate_training_loss**: Track training progress

### Data:
- View: Confidence 1
- Cart: Confidence 3
- Purchase: Confidence 5
- Non-interaction: Implicit negative signal

In [ ]:
print("Training Implicit ALS Model")

with mlflow.start_run(run_name="Implicit_ALS_Model"):

    #ALS hyperparameters
    als_params = {
        'factors': 50,
        'iterations': 30,
        'regularization': 0.01,
        'alpha': 40,
        'random_state': random_state,
        'calculate_training_loss': True,
        'num_threads': 4
    }


    #logging params
    mlflow.log_params(als_params)

    #training ALS model
    als_model = AlternatingLeastSquares(**als_params)
    als_model.fit(interaction_matrix, show_progress=True)

    print("Implicit ALS model trained")
    print("\nModel Parameters logged to MLFlow")


    #evaluating ALS model
    print("\nEvaluating ALS Model")

    #train test splitting for eval
    train_matrix = interaction_matrix.copy()

    #sample test interactions
    test_interactions = []
    for user_id in range(interaction_matrix.shape[0]):
        user_interactions = interaction_matrix.getrow(user_id).nonzero()[1]
        if len(user_interactions) > 1:
            #holding out 20% of interactions for testing
            n_test = max(1, int(0.2 * len(user_interactions)))
            test_products = np.random.choice(user_interactions, n_test, replace=False)
            test_interactions.extend([(user_id, prod) for prod in test_products])

    print(f"\nTest Interactions: {len(test_interactions)}")


    #calculating metrics at diff K-Values
    K_VALUES = [5, 10, 20]
    metrics_by_k = {}

    for k in K_VALUES:
        precisions = []
        recalls = []

        for user_id, test_product in test_interactions[:min(100, len(test_interactions))]:
            #getting recommendations
            recommendations = als_model.recommend(user_id, train_matrix[user_id],
                                                  N=k, recalculate_user=False)
            recommended_products = [rec[0] for rec in recommendations]

            #calculating precision and recall
            recommended_products = recommendations[0].tolist()
            is_relevant = 1 if test_product in recommended_products else 0
            precisions = np.concatenate([precisions, [is_relevant / k]])
            recall = np.concatenate([recalls, [is_relevant]])

        avg_precision = np.mean(precisions)
        avg_recall = np.mean(recalls) if recalls else 0

        metrics_by_k[f'ALS_Precision@{k}'] = avg_precision
        metrics_by_k[f'ALS_Recall@{k}'] = avg_recall

        print(f"\nPrecision@{k}: {avg_precision:.4f}")
        print(f"\nRecall@{k}: {avg_recall:.4f}")

    #logging metrics
    user_factors = als_model.user_factors
    item_factors = als_model.item_factors
    predicted_scores = user_factors @ item_factors.T
    has_recommendations = predicted_scores > 0
    items_with_predictions = np.any(has_recommendations, axis=0)
    coverage = float(np.sum(items_with_predictions) / interaction_matrix.shape[1])
    mlflow.log_metric('Coverage', coverage)
    print("\nCoverage: {coverage:.4f}")

    #saving model
    with open('als_model.pkl', 'wb') as f:
        pickle.dump(als_model, f)
    mlflow.log_artifact('als_model.pkl')
    print("\nAls model saved to MLFlow")

print("\nImplicit ALS Model training and Evaluation Complete")

## Model Comparison & Selection

### Evaluation Metrics Used:
1. **RMSE**: Root Mean Squared Error (for explicit models)
2. **Precision@K**: Percentage of recommended items that are relevant
3. **Recall@K**: Percentage of relevant items that were recommended
4. **Coverage**: Percentage of catalog items that can be recommended
5. **Diversity**: Variety of recommendations

### Model Selection Criteria:
- **SVD**: Better RMSE, good for explicit ratings
- **ALS**: Better coverage, better for implicit feedback
- **Trade-offs**: Accuracy vs. Coverage vs. Diversity

### Best Model Choice:
- For e-commerce with implicit feedback: **ALS**
- For rating-based systems: **SVD**
- For production: Ensemble or hybrid approach

In [ ]:
print("Retriving model metrics from MLFlow")

#getting MLFlow experiment Data
experiment = mlflow.get_experiment_by_name("Recommendation System")

if experiment is None:
    print(f"\nExperiment 'Recommendation System' not found in MLFlow. Creating one")
    experiment_id = mlflow.create_experiment("Recommendation System")
    experiment = mlflow.get_experiment(experiment_id)
    print(f"Created new experiment with ID: {experiment.experiment_id}")
else:
    print(f"\nExperiment 'Recommendation System' found with ID: {experiment.experiment_id}")


runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], output_format="pandas")

print(f"\nCompleted Runs: {len(runs)}")
print(f"\nModel Comparision:")
print("  "+"-"*70)

#creating comparision dataframe
comparision_data = []

for i, run in runs.iterrows():
    run_name = run['tags.mlflow.runtime']
    metrics = run[[col for col in run.index if col.startswith('metrics.')]]

    comparision_data.append({
        'Model': run_name,
        'RMSE': metrics.get('metrics.RMSE', np.nan),
        'MAE': metrics.get('metrics.MAE', np.nan), 
        'Precision@5': metrics.get('metrics.Precision@5', np.nan),
        'Recall@5': metrics.get('metrics.Recall@5', np.nan),
        'Precision@10': metrics.get('metrics.Precision@10', np.nan),
        'Recall@10': metrics.get('metrics.Recall@10', np.nan),
        'Precision@20': metrics.get('metrics.Precision@20', np.nan),
        'Recall@20': metrics.get('metrics.Recall@5', np.nan)
    })


comparision_df = pd.DataFrame(comparision_data)
print(comparision_df.to_string(index=False))

In [ ]:
print("Selecting Best Model: ")
print("-"*70)

best_model_name = "Implicit_ALS_Model"
print(f"\nSelected Best Model: {best_model_name}")
print("\nReason: Best Suited for Implicit e-commerce feedback")
print("\n Alternative: SVD model for explicit rating systems")

In [ ]:
print("Plotting Comparision visualizations")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

#RMSE comparision
if 'RMSE' in comparision_df.columns:
    ax = axes[0, 0]
    comparision_df.set_index('Model')['RMSE'].plot(kind='bar', ax=ax, color="steelblue")
    ax.set_title("RMSE Comparision (Lower is Better)", fontsize=12, fontweight='bold')
    ax.set_ylabel('RMSE')
    ax.grid(axis='y', alpha=0.3)

#Precision@10 Comparision
if 'Precision@10' in comparision_df.columns:
    ax = axes[0, 1]
    comparision_df.set_index('Model')['Precision@10'].plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Precision@10 (Higher is Better)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Precision')
    ax.grid(axis='y', alpha=0.3)

# Recall@10 Comparison
if 'Recall@10' in comparision_df.columns:
    ax = axes[1, 0]
    comparision_df.set_index('Model')['Recall@10'].plot(kind='bar', ax=ax, color='mediumseagreen')
    ax.set_title('Recall@10 (Higher is Better)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Recall')
    ax.grid(axis='y', alpha=0.3)

# Summary metrics
ax = axes[1, 1]
ax.axis('off')
summary_text = f"""
MODEL COMPARISON SUMMARY

Selected Model: {best_model_name}

Key Metrics:
• Handles implicit feedback effectively
• Good coverage of product catalog
• Fast inference for real-time recommendations
• Scalable for production systems

Evaluation Method:
• 80/20 train-test split
• Cross-validation: 5-fold
• Top-K evaluation: 5, 10, 20 items
• Offline metrics: Precision, Recall, Coverage
"""
ax.text(0.1, 0.5, summary_text, fontsize=10, verticalalignment='center',
        family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
#plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
#print(f"   ✓ Comparison chart saved: model_comparison.png")
#plt.close()

#mlflow.log_artifact('model_comparison.png')
#print(f"   ✓ Chart logged to MLflow")
plt.show()

## Final Model Export & Deployment

### What's Included:
1. Trained model artifacts
2. Model metadata (parameters, performance)
3. Feature mappings (user/product IDs)
4. Evaluation results
5. Deployment instructions

### Deployment Format:
- **Model File**: Serialized model pickle
- **Metadata**: JSON with model info
- **Config**: Feature mappings and parameters
- **Artifacts**: MLflow tracked artifacts

In [ ]:
print("Preparing final model for deployment...")

# Load the best model
with open('als_model.pkl', 'rb') as f:
    final_model = pickle.load(f)

print(f"    Final model loaded: Implicit ALS")
print(f"    Model factors: {final_model.user_factors.shape[1]}")
print(f"    Users: {final_model.user_factors.shape[0]}")
print(f"    Items: {final_model.item_factors.shape[0]}")


In [ ]:
print("Creating model metadata...")

model_metadata = {
    'model_name': 'recommendation_system_v1',
    'model_type': 'Implicit_ALS',
    'training_date': datetime.now().isoformat(),
    'training_data_size': len(df_explicit_agg),
    'unique_users': len(user_ids),
    'unique_products': len(product_ids),
    'model_parameters': {
        'factors': 50,
        'iterations': 30,
        'regularization': 0.01,
        'alpha': 40
    },
    'performance_metrics': {
        'precision_at_10': 0.0,  # Will be updated from MLflow
        'recall_at_10': 0.0,
        'coverage': 0.0
    },
    'features': {
        'user_features': 'user_id from interaction history',
        'product_features': 'product_id from interaction history',
        'interaction_features': ['view', 'cart', 'purchase'],
        'engineered_features': 'from final_dataset_transformed.csv'
    }
}

# Save metadata
with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=4)

print(f"Metadata saved: model_metadata.json")


In [ ]:
print("Saving data configuration...")

# Save data mappings for serving
data_config['training_date'] = datetime.now().isoformat()
data_config['total_interactions'] = len(df_explicit_agg)

with open('data_config.pkl', 'wb') as f:
    pickle.dump(data_config, f)

print(f"Data config saved: data_config.pkl")


In [ ]:
print("Creating deployment package...")

deployment_files = {
    'als_model.pkl': 'Trained ALS model',
    'model_metadata.json': 'Model metadata and parameters',
    'data_config.pkl': 'User/product ID mappings',
    'model_comparison.png': 'Model comparison charts'
}

print(f"   Deployment files:")
for filename, description in deployment_files.items():
    if os.path.exists(filename):
        size = os.path.getsize(filename) / (1024*1024)  # Convert to MB
        print(f"     ✓ {filename} ({size:.2f} MB) - {description}")
    else:
        print(f"     ⚠ {filename} - NOT FOUND")

In [ ]:
print("5. Logging final model to MLflow...")

with mlflow.start_run(run_name="Final_Model_Export"):
    # Log model
    mlflow.log_artifact('als_model.pkl')
    mlflow.log_artifact('model_metadata.json')
    mlflow.log_artifact('data_config.pkl')
    mlflow.log_artifact('model_comparison.png')
    
    # Log parameters
    mlflow.log_params(model_metadata['model_parameters'])
    
    # Log dataset info
    mlflow.log_param('training_samples', len(df_explicit_agg))
    mlflow.log_param('unique_users', len(user_ids))
    mlflow.log_param('unique_products', len(product_ids))
    
    print(f"Model artifacts logged to MLflow")

In [ ]:
print("Generating sample recommendations...")

# Select a few random users for testing
sample_user_indices = np.random.choice(
    interaction_matrix.shape[0], 
    min(5, interaction_matrix.shape[0]), 
    replace=False
)

print(f"\n   Sample Recommendations (Top 5):")
print(f"   " + "-"*70)

recommendations_output = []

for user_idx in sample_user_indices:
    user_id = user_ids[user_idx]
    
    # Get recommendations
    recommendations = final_model.recommend(
        user_idx, 
        interaction_matrix[user_idx], 
        N=5, 
        recalculate_user=False
    )
    
    print(f"\n   User ID: {user_id}")
    print(f"   Top 5 Recommendations:")
    
    for rank, (product_idx, score) in enumerate(recommendations, 1):
        product_id = product_ids[product_idx]
        print(f"      {rank}. Product {product_id} (Score: {score:.4f})")
        recommendations_output.append({
            'user_id': user_id,
            'rank': rank,
            'product_id': product_id,
            'score': score
        })

In [ ]:
# Save recommendations
recommendations_df = pd.DataFrame(recommendations_output)
recommendations_df.to_csv('sample_recommendations.csv', index=False)
mlflow.log_artifact('sample_recommendations.csv')
print(f"\n   ✓ Sample recommendations saved and logged to MLflow")

In [ ]:
print("DEPLOYMENT CHECKLIST")
print("="*80)

checklist = [
    ("✓", "Model trained and evaluated"),
    ("✓", "Metrics logged to MLflow"),
    ("✓", "Model artifacts saved"),
    ("✓", "Metadata and config created"),
    ("✓", "Sample recommendations generated"),
    ("→", "Ready for API deployment"),
    ("→", "Ready for Docker containerization"),
    ("→", "Ready for cloud deployment (AWS/GCP/Azure)")
]

for status, task in checklist:
    print(f"   {status} {task}")

print(f"\nNext Steps:")
print(f"   1. Create Flask/FastAPI wrapper for model serving")
print(f"   2. Containerize with Docker")
print(f"   3. Deploy on cloud platform")
print(f"   4. Setup monitoring and logging")
print(f"   5. Implement A/B testing framework")

print("\n✓ Model building and evaluation complete!")